In [1]:
import importlib
import torch
import torch.nn.functional as F
from matplotlib import pyplot as plt
from PIL import Image

import diffusion_pt.models.unet
from diffusion_pt.gpu_utils.datasets import get_dataloader

# Reload the specific module
importlib.reload(diffusion_pt.models.unet)

# Import the model again
from diffusion_pt.models.unet import UNet2DModel

from diffusion_pt.diffusion_utils import get_beta_schedule, GaussianDiffusion

In [2]:
torch.manual_seed(5)

In [3]:
from dataclasses import dataclass
import os

log_dir = 'logs'
model_dir = os.path.join(log_dir, 'models', 'celebhq_diffusion')

@dataclass
class TrainingConfig:
    image_size = 256  # the generated image resolution
    train_batch_size = 4
    eval_batch_size = 4  # how many images to sample during evaluation
    num_epochs = 2
    gradient_accumulation_steps = 1
    learning_rate = 2e-5
    lr_warmup_steps = 1000
    save_image_epochs = 1
    save_model_epochs = 2
    mixed_precision = 'no'  # `no` for float32, `fp16` for automatic mixed precision
    output_dir = model_dir  # the model namy locally and on the HF Hub

    push_to_hub = False  # whether to upload the saved model to the HF Hub
    hub_private_repo = True  
    overwrite_output_dir = True  # overwrite the old model when re-running the notebook
    seed = 5

config = TrainingConfig()

In [4]:
data_dir = '../datasets/celebahq256/celeba_hq_256/'
dataset_name = 'celebahq256'

dataloader = get_dataloader(
    name=dataset_name,
    data_dir=data_dir,
    batch_size=config.train_batch_size,
    shuffle=True,
    num_workers=4
)

In [5]:
num_diffusion_timesteps = 1000
beta_start = 0.0001
beta_end = 0.02
beta_schedule = 'linear'
loss_type = 'noisepred'
dropout = 0.0
randflip = 1
block_size = 1,
log_dir = 'logs'
model_name='unet2d16b2c112244'

model = UNet2DModel(
            num_classes=1,
            in_channels=3,  # Assuming RGB images
            ch=128,
            num_res_blocks=2,
            initial_resolution=256,
            attn_resolutions=(16,),
            out_ch=3,
            ch_mult=(1, 1, 2, 2, 4, 4),
            dropout=dropout,
            resamp_with_conv=True,
            conv_shortcut=True
        )

# diffusion = GaussianDiffusion(
#             betas=get_beta_schedule(
#                 beta_schedule, beta_start=beta_start, beta_end=beta_end, num_diffusion_timesteps=num_diffusion_timesteps
#             ),
#             loss_type=loss_type,
#             device='cuda:0'  # Pass the device here
#         )

In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

In [7]:
from diffusers.optimization import get_cosine_schedule_with_warmup

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=config.lr_warmup_steps,
    num_training_steps=(len(dataloader) * config.num_epochs)
)

/home/abhishek/anaconda3/envs/py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import torch
from PIL import Image
import os

def make_grid(images, rows, cols):
    w, h = images[0].size
    grid = Image.new('RGB', size=(cols * w, rows * h))
    for i, image in enumerate(images):
        grid.paste(image, box=(i % cols * w, i // cols * h))
    return grid

def evaluate(config, epoch, diffusion: GaussianDiffusion, unet):
    def denoise(x, t, dropout=0.0):
        B, C, H, W = x.shape
        assert x.dtype == torch.float32
        assert t.shape == (B,)
        
        # Move x and t to the same device as the model
        device = next(unet.parameters()).device
        x = x.to(device)
        t = t.to(device)
        
        out = unet(x, t)
        assert out.shape == (B, C, H, W)
        return out

    # Define the noise tensor with 4 samples
    # t = torch.full((4,), 1000, device='cuda', dtype=torch.int32)

    device = next(unet.parameters()).device
    assert device == diffusion.device, f"unet device: {device}, diffusion device: {diffusion.device}"
    # Generate images using the diffusion model
    with torch.no_grad():
        images = diffusion.p_sample_loop(
            denoise_fn=lambda x, t: denoise(x, t, dropout=0.0),
            shape=torch.Size([2, 3, 256, 256]),
            device=device
        )

    # Samples are in [-1, 1]; scale to [0, 1] for visualization
    images = (images + 1) / 2
    images = images.clamp(0, 1)

    # Convert PyTorch tensors to PIL Images
    images = [Image.fromarray((img.permute(1, 2, 0).cpu().numpy() * 255).astype('uint8')) for img in images]

    # Make a grid of the generated images
    image_grid = make_grid(images, rows=2, cols=2)

    # Save the image grid
    test_dir = os.path.join(config.output_dir, "samples")
    os.makedirs(test_dir, exist_ok=True)
    image_grid.save(f"{test_dir}/{epoch:04d}.png")
    print(f"saved sample images at: {test_dir}/{epoch:04d}.png")


In [9]:
from accelerate import Accelerator
from tqdm.auto import tqdm
from pathlib import Path

In [10]:
def train_loop(config: TrainingConfig, model: UNet2DModel, optimizer, train_dataloader, lr_scheduler):
    # Initialize accelerator and tensorboard logging
    accelerator = Accelerator(
        mixed_precision=config.mixed_precision,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        log_with="tensorboard",
        project_dir=os.path.join(config.output_dir, "logs")
    )
    if accelerator.is_main_process:
        if config.output_dir is not None:
            os.makedirs(config.output_dir, exist_ok=True)
    accelerator.init_trackers("train_example")
    
    # Prepare everything
    # There is no specific order to remember, just need to unpack the 
    # objects in the same order as in input to the prepare method.
    model, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
        model, optimizer, train_dataloader, lr_scheduler
    )
    
    device = next(model.parameters()).device
    noise_scheduler = GaussianDiffusion(
            betas=get_beta_schedule(
                beta_schedule, beta_start=beta_start, beta_end=beta_end, num_diffusion_timesteps=num_diffusion_timesteps
            ),
            loss_type=loss_type,
            device=device  # Pass the device here
        )
    
    global_step = 0
    
    # Now train the model
    for epoch in range(config.num_epochs):
        progress_bar = tqdm(total=len(train_dataloader), disable=not accelerator.is_local_main_process)
        progress_bar.set_description(f"Epoch {epoch}")
        
        for step, batch in enumerate(train_dataloader):
            clean_images, y = batch['image'], batch['label']
            noise = torch.randn(clean_images.shape).to(clean_images.device)
            bs = clean_images.shape[0]
            
            timesteps = torch.randint(0, noise_scheduler.num_timesteps, (bs,), device=clean_images.device, dtype=torch.int32)
            
            noisy_images = noise_scheduler.q_sample(x_start=clean_images, t=timesteps, noise=noise)
            
            with accelerator.accumulate(model):
                noise_pred = model(noisy_images, timesteps)
                loss = F.mse_loss(noise_pred, noise)
                accelerator.backward(loss)
                
                accelerator.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()
                
            progress_bar.update(1)
            logs = {"loss": loss.detach().item(), "lr": lr_scheduler.get_last_lr()[0], "step": global_step}
            progress_bar.set_postfix(**logs)
            accelerator.log(logs, step=global_step)
            global_step += 1
            
            # After each epoch optionally sample some demo images with evaluate() and save the model
            if accelerator.is_main_process:
                evaluate(config, epoch, noise_scheduler, unet=accelerator.unwrap_model(model))
                break
            break
            

In [11]:
from accelerate import notebook_launcher
args = (config, model, optimizer, dataloader, lr_scheduler)

notebook_launcher(train_loop, args, num_processes=1)

Launching training on one GPU.


Epoch 0:   0%|          | 1/7500 [00:01<2:25:15,  1.16s/it, loss=1.12, lr=2e-8, step=0]

saved sample images at: logs/models/celebhq_diffusion/samples/0000.png


Epoch 1:   0%|          | 1/7500 [02:06<263:54:55, 126.70s/it, loss=1.13, lr=4e-8, step=1]

saved sample images at: logs/models/celebhq_diffusion/samples/0001.png
